# PropertyData spike

A `PropertyData` is a JAX array whose last axis is aligned with a `PropertyMap`.
Selectors, partitions, and reductions reuse the existing taxonomy queries;
the array is the only pytree leaf. This notebook asserts the algebra and
measures jit dispatch, scoped updates, `segment_sum` vs one-hot, and whether
XLA already fuses elementwise chains (the case against a lazy layer).

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src" / "summer4").is_dir():
    ROOT = Path(__file__).resolve().parents[2] if "__file__" in dir() else ROOT
sys.path.insert(0, str(ROOT))

from explorations.datatypes.prototype import (
    VARIANT_DIGEST,
    VARIANT_IDENTITY,
    DigestStatic,
    IdentityStatic,
    PropertyData,
)
from summer4 import Property, PropertyMap

print("jax", jax.__version__)

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
sev = Property("severity", ("mild", "severe"))
pm = PropertyMap.from_property(state).stratify(age).stratify(sev, where=state["I"])
assert pm.size == 12

values = jnp.arange(pm.size, dtype=jnp.float32)
d = PropertyData.wrap(pm, values)
assert d.data.shape == (12,)

## Algebra

Elementwise ops, selector gather/scatter, reduce along a property, and the
modelling composition `normalise_within`.

In [ ]:
assert jnp.allclose((d * 2 + 1).data, values * 2 + 1)
assert jnp.allclose((3 * d).data, values * 3)
assert jnp.allclose((d + d).data, values * 2)

infected = d[state["I"]]
assert infected.shape == (pm.select(state["I"]).size,)
assert jnp.allclose(infected, values[pm.select(state["I"])])

zeroed = d.where(state["S"], 0.0)
assert jnp.allclose(zeroed[state["S"]], 0.0)
assert jnp.allclose(zeroed[state["I"]], values[pm.select(state["I"])])

bumped = d.at[state["I"]].add(10.0)
assert jnp.allclose(bumped[state["I"]], values[pm.select(state["I"])] + 10.0)
assert jnp.allclose(bumped[state["S"]], values[pm.select(state["S"])])

sub = d.subset(state["I"] & age["0-4"])
assert sub.pmap.size == 2
assert sub.data.shape == (2,)

by_age = d.partition(age)
assert set(t.name for t in by_age) == {"0-4", "5-9", "10+"}
assert sum(int(v.size) for v in by_age.values()) == pm.size

In [ ]:
ones = PropertyData.wrap(pm, jnp.ones(pm.size, dtype=jnp.float32))
by_age_sum = ones.sum_over(age)
assert by_age_sum.pmap.size == 3
assert by_age_sum.pmap.properties[0].name == "age"
# Each age appears on S, I-mild, I-severe, R → 4 compartments.
assert jnp.allclose(by_age_sum.data, jnp.array([4.0, 4.0, 4.0]))

via_onehot = ones.sum_over_onehot(age)
assert jnp.allclose(by_age_sum.data, via_onehot.data)

expanded = ones.broadcast_over(age, by_age_sum)
assert expanded.data.shape == (pm.size,)
assert jnp.allclose(expanded.data, 4.0)

norm = ones.normalise_within(age)
assert jnp.allclose(norm.data, 0.25)
for idx in pm.partition(age).values():
    assert jnp.allclose(norm.data[idx].sum(), 1.0)

## Pytree and jit

`tree_map` unflattens with arbitrary leaves, so `PropertyData` must not check
shapes in `__init__`. Both hashing variants must survive `jit`.

In [ ]:
doubled = jax.tree.map(lambda x: x * 2, d)
assert isinstance(doubled, PropertyData)
assert doubled.pmap == d.pmap
assert jnp.allclose(doubled.data, values * 2)


class Sentinel:
    shape = None


structure = jax.tree.structure(d)
rebuilt = jax.tree.unflatten(structure, [Sentinel()])
assert rebuilt.pmap == d.pmap
assert isinstance(rebuilt.data, Sentinel)

d.check()


@jax.jit
def scale(pd):
    return (pd * 2 + 1).at[state["I"]].add(0.5)


out = scale(d)
expect = values * 2 + 1
expect = expect.at[pm.select(state["I"])].add(0.5)
assert jnp.allclose(out.data, expect)

d_id = PropertyData.wrap(pm, values, variant=VARIANT_IDENTITY)
out_id = scale(d_id)
assert jnp.allclose(out_id.data, expect)

clone = PropertyMap(
    properties=pm.properties, codes=np.array(pm.codes), history=pm.history, parent_row=pm.parent_row
)
assert clone == pm
assert hash(DigestStatic(clone)) == hash(DigestStatic(pm))
assert DigestStatic(clone) == DigestStatic(pm)
assert IdentityStatic(clone) != IdentityStatic(pm)

## Benchmarks

1k (`3 × 10 × 34 = 1020`) and 100k (`10^5`) cartesian maps. Times are median
seconds over a small host loop; device work is `block_until_ready`.

In [ ]:
def _sir_like(n_age: int, n_loc: int) -> PropertyMap:
    st = Property("state", ("S", "I", "R"))
    ag = Property("age", tuple(f"a{i}" for i in range(n_age)))
    loc = Property("loc", tuple(f"l{i}" for i in range(n_loc)))
    return PropertyMap.from_property(st).stratify(ag).stratify(loc)


def _cartesian(factors: tuple[int, ...]) -> PropertyMap:
    props = [Property(f"p{i}", tuple(f"t{j}" for j in range(n))) for i, n in enumerate(factors)]
    built = PropertyMap.from_property(props[0])
    for prop in props[1:]:
        built = built.stratify(prop)
    return built


def _ready(out):
    if hasattr(out, "block_until_ready"):
        out.block_until_ready()
    elif isinstance(out, PropertyData):
        out.data.block_until_ready()
    return out


def bench(fn, *, n=25, warmup=3, name=""):
    for _ in range(warmup):
        _ready(fn())
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        _ready(fn())
        times.append(time.perf_counter() - t0)
    times.sort()
    med = times[len(times) // 2]
    print(f"{name:48s}  median={med * 1e6:9.1f} us")
    return med


RESULTS: dict[str, float] = {}


def record(key: str, value: float) -> None:
    RESULTS[key] = value

In [ ]:
def run_size(label: str, pmap: PropertyMap, *, repeats: int) -> None:
    x = jnp.arange(pmap.size, dtype=jnp.float32)
    pd_a = PropertyData.wrap(pmap, x, variant=VARIANT_DIGEST)
    pd_b = PropertyData.wrap(pmap, x, variant=VARIANT_IDENTITY)
    pd_a2 = PropertyData.wrap(
        PropertyMap(
            properties=pmap.properties,
            codes=np.array(pmap.codes),
            history=pmap.history,
            parent_row=pmap.parent_row,
        ),
        x,
        variant=VARIANT_DIGEST,
    )
    pd_b2 = PropertyData.wrap(
        PropertyMap(
            properties=pmap.properties,
            codes=np.array(pmap.codes),
            history=pmap.history,
            parent_row=pmap.parent_row,
        ),
        x,
        variant=VARIANT_IDENTITY,
    )
    sel = pmap.properties[0][pmap.properties[0].traits[0]]
    idx = pmap.select(sel)
    prop0 = pmap.properties[0]

    @jax.jit
    def raw_mul(arr):
        return arr * 2.0

    @jax.jit
    def wrap_mul(pd):
        return pd * 2.0

    @jax.jit
    def raw_add_at(arr):
        return arr.at[idx].add(1.0)

    @jax.jit
    def wrap_add_at(pd):
        return pd.at[sel].add(1.0)

    @jax.jit
    def wrap_sum(pd):
        return pd.sum_over(prop0)

    @jax.jit
    def wrap_onehot(pd):
        return pd.sum_over_onehot(prop0)

    @jax.jit
    def chained(pd):
        return ((pd * 1.1) + 0.2) * 0.9 - 0.05

    @jax.jit
    def fused(pd):
        return pd * (1.1 * 0.9) + (0.2 * 0.9 - 0.05)

    raw_mul(x)
    wrap_mul(pd_a)
    wrap_mul(pd_b)
    raw_add_at(x)
    wrap_add_at(pd_a)
    wrap_sum(pd_a)
    wrap_onehot(pd_a)
    chained(pd_a)
    fused(pd_a)

    record(f"{label}/raw_mul", bench(lambda: raw_mul(x), n=repeats, name=f"{label} raw *2"))
    record(
        f"{label}/digest_mul", bench(lambda: wrap_mul(pd_a), n=repeats, name=f"{label} digest *2")
    )
    record(
        f"{label}/identity_mul",
        bench(lambda: wrap_mul(pd_b), n=repeats, name=f"{label} identity *2"),
    )
    record(
        f"{label}/digest_equal_clone",
        bench(lambda: wrap_mul(pd_a2), n=repeats, name=f"{label} digest clone *2"),
    )
    # First call on a new identity-wrapped equal map retraces.
    t0 = time.perf_counter()
    _ready(wrap_mul(pd_b2))
    record(f"{label}/identity_clone_first", time.perf_counter() - t0)
    print(
        f"{label + ' identity clone first':48s}  {RESULTS[f'{label}/identity_clone_first'] * 1e3:9.2f} ms"
    )

    record(f"{label}/raw_at", bench(lambda: raw_add_at(x), n=repeats, name=f"{label} raw at.add"))
    record(
        f"{label}/wrap_at", bench(lambda: wrap_add_at(pd_a), n=repeats, name=f"{label} wrap at.add")
    )
    record(
        f"{label}/segment_sum",
        bench(lambda: wrap_sum(pd_a), n=repeats, name=f"{label} segment_sum"),
    )
    record(
        f"{label}/onehot",
        bench(lambda: wrap_onehot(pd_a), n=repeats, name=f"{label} onehot matmul"),
    )
    record(
        f"{label}/chained", bench(lambda: chained(pd_a), n=repeats, name=f"{label} chained elem")
    )
    record(f"{label}/fused", bench(lambda: fused(pd_a), n=repeats, name=f"{label} fused elem"))

    got = wrap_sum(pd_a).data
    # float32 reduction order differs between segment_sum and matmul.
    np.testing.assert_allclose(
        np.asarray(got), np.asarray(wrap_onehot(pd_a).data), rtol=1e-4, atol=1.0
    )
    np.testing.assert_allclose(
        np.asarray(chained(pd_a).data),
        np.asarray(fused(pd_a).data),
        rtol=1e-4,
        atol=1e-3,
    )


run_size("1k", _sir_like(10, 34), repeats=30)
run_size("100k", _cartesian((10, 10, 10, 10, 10)), repeats=15)
print("RESULTS", {k: round(v * 1e6, 2) for k, v in RESULTS.items()})

In [ ]:
assert RESULTS["1k/digest_mul"] > 0
assert RESULTS["1k/identity_mul"] > 0
assert RESULTS["100k/chained"] > 0
assert RESULTS["100k/fused"] > 0
# Fusion hypothesis: chained and fused should be within a small factor.
ratio = RESULTS["100k/chained"] / RESULTS["100k/fused"]
print(f"100k chained/fused ratio = {ratio:.3f}")
assert 0.25 < ratio < 4.0